In [ ]:
import os, random, glob
from pathlib import Path

IMAGES_DIR = "/kaggle/input/datasets/dimabimov/aaa-projec/dataset/images"
if not os.path.isdir(IMAGES_DIR):
    cands = sorted(glob.glob("/kaggle/input/**/images", recursive=True))
    if cands:
        IMAGES_DIR = cands[0]
print("IMAGES_DIR:", IMAGES_DIR, "| exists:", os.path.isdir(IMAGES_DIR))

OUT_DIR = Path("/kaggle/working")

# Две готовые разметки (залей как Kaggle-датасеты, пропиши пути):
#   TWO_CLASS_CSV  - real_estate/floor_plan, колонки filename + (pred_class|label), опц. confidence.
#   SCREENSHOT_CSV - screenshot из find_black_bars, колонка filename.
TWO_CLASS_CSV = "/kaggle/input/photo-type-labels/siglip2_labels.csv"
SCREENSHOT_CSV = "/kaggle/input/black-bar-screenshots/black_bar_screenshots.csv"

CLASSES = ["real_estate", "floor_plan", "screenshot"]
SEED = 42
CONF_THRESHOLD = 0.85    # порог для двух классов, если в CSV есть колонка confidence
MAX_PER_CLASS = 4000
IMG_SIZE = 224
device = "cuda"
random.seed(SEED)


In [ ]:
import torch

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    arch = f"sm_{cap[0]}{cap[1]}"
    print("GPU:", torch.cuda.get_device_name(0), arch, "| torch:", torch.__version__)
    if arch not in torch.cuda.get_arch_list():
        raise SystemExit(
            f"\nGPU {torch.cuda.get_device_name(0)} ({arch}) несовместим с torch на Kaggle.\n"
            "Settings (панель справа) -> Accelerator -> GPU T4 x2, дождись перезапуска -> Run All."
        )
else:
    raise SystemExit("CUDA недоступна. Settings -> Accelerator -> GPU T4 x2, затем Run All.")


In [ ]:
import os, glob
import pandas as pd
from sklearn.model_selection import train_test_split

# индекс basename -> полный путь (терпит подпапки и любую раскладку датасета)
all_files = glob.glob(os.path.join(IMAGES_DIR, "**", "*"), recursive=True)
name2path = {os.path.basename(p): p for p in all_files if os.path.isfile(p)}
print("картинок под IMAGES_DIR:", len(name2path))
assert name2path, f"под IMAGES_DIR={IMAGES_DIR} не найдено картинок, проверь путь к датасету"

def resolve(frame):
    frame = frame.copy()
    frame["path"] = frame["filename"].astype(str).map(name2path)
    n0 = len(frame)
    frame = frame[frame["path"].notna()].reset_index(drop=True)
    print(f"  resolved {len(frame)}/{n0}")
    return frame

# screenshot - из ручной разметки
ss = pd.read_csv(SCREENSHOT_CSV)
print("SCREENSHOT_CSV строк:", len(ss), "| колонки:", list(ss.columns))
ss_files = set(ss["filename"].astype(str))
print("screenshot:")
ss_df = resolve(pd.DataFrame({"filename": sorted(ss_files), "pred_class": "screenshot"}))

# real_estate / floor_plan - из второй разметки, исключая файлы-скриншоты
two = pd.read_csv(TWO_CLASS_CSV)
if "pred_class" not in two.columns and "label" in two.columns:
    two = two.rename(columns={"label": "pred_class"})
print("TWO_CLASS_CSV строк:", len(two), "| классы:", two["pred_class"].value_counts().to_dict())
if "confidence" in two.columns:
    two = two[two["confidence"] >= CONF_THRESHOLD]
two = two[two["pred_class"].isin(["real_estate", "floor_plan"])]
two = two[~two["filename"].astype(str).isin(ss_files)]
print("real_estate/floor_plan:")
two = resolve(two[["filename", "pred_class"]])

parts = [ss_df.sample(min(len(ss_df), MAX_PER_CLASS), random_state=SEED)]
for c in ["real_estate", "floor_plan"]:
    s = two[two.pred_class == c]
    parts.append(s.sample(min(len(s), MAX_PER_CLASS), random_state=SEED))
train_df = pd.concat(parts).reset_index(drop=True)
print(train_df.pred_class.value_counts())
assert len(train_df) > 0, "trainset пуст: filename из CSV не совпали с файлами датасета (см. resolved выше)"

tr, va = train_test_split(train_df, test_size=0.15, stratify=train_df.pred_class, random_state=SEED)
print("train:", len(tr), "| val:", len(va))


In [ ]:
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Без кропа: у скриншотов чёрные полосы по краям, Resize их сохраняет (это нужный признак).
cls2idx = {c: i for i, c in enumerate(CLASSES)}
_norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.1, 0.0),
    T.ToTensor(), _norm,
])
val_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), _norm])

class ClsDataset(Dataset):
    def __init__(self, frame, tf):
        self.f = frame.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        return self.tf(img), cls2idx[r["pred_class"]]

counts = tr.pred_class.value_counts().to_dict()
weights = tr.pred_class.map(lambda c: 1.0 / counts[c]).values
sampler = WeightedRandomSampler(torch.DoubleTensor(weights), num_samples=len(weights), replacement=True)
train_loader = DataLoader(ClsDataset(tr, train_tf), batch_size=64, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(ClsDataset(va, val_tf), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
import timm
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler

net = timm.create_model("mobilenetv3_small_100", pretrained=True, num_classes=len(CLASSES)).to(device)
EPOCHS = 12
opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_loader))
crit = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = GradScaler()

@torch.no_grad()
def evaluate():
    net.eval()
    ys, ps = [], []
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        with autocast():
            out = net(x)
        ps.append(out.argmax(1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

history = []
for ep in range(EPOCHS):
    net.train()
    run = 0.0
    for x, y in tqdm(train_loader, desc=f"epoch {ep+1}/{EPOCHS}"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with autocast():
            loss = crit(net(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item() * x.size(0)
    yv, pv = evaluate()
    acc = float((yv == pv).mean())
    history.append((ep + 1, run / len(tr), acc))
    print(f"epoch {ep+1}: train_loss={run/len(tr):.4f}  val_acc={acc:.4f}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

yv, pv = evaluate()
print(classification_report(yv, pv, target_names=CLASSES, digits=3))
cm = confusion_matrix(yv, pv)
print("confusion matrix (rows=true label, cols=pred CNN):")
print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))


In [ ]:
import matplotlib.pyplot as plt

# кривая обучения
if history:
    eps = [h[0] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(eps, [h[1] for h in history], marker="o")
    ax[0].set_title("train loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)
    ax[1].plot(eps, [h[2] for h in history], marker="o", color="green")
    ax[1].set_title("val accuracy"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "training_curve.png", dpi=110); plt.show()

# галерея: что модель к какому классу относит (предсказываем на сэмпле валидации, группируем по предсказанию)
net.eval()
pool = va.sample(min(120, len(va)), random_state=SEED).reset_index(drop=True)
preds = []
with torch.no_grad():
    for _, r in pool.iterrows():
        img = Image.open(r["path"]).convert("RGB")
        with autocast():
            out = net(val_tf(img).unsqueeze(0).to(device))
        preds.append(CLASSES[out.argmax(1).item()])
pool["pred"] = preds

per_class = 5
fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(per_class * 2.6, len(CLASSES) * 2.9))
for ri, c in enumerate(CLASSES):
    sub = pool[pool.pred == c].head(per_class).reset_index(drop=True)
    for ci in range(per_class):
        ax = axes[ri][ci]
        ax.axis("off")
        if ci == 0:
            ax.text(-0.15, 0.5, f"pred:\n{c}", transform=ax.transAxes,
                    rotation=90, va="center", ha="center", fontsize=11, fontweight="bold")
        if ci >= len(sub):
            continue
        r = sub.iloc[ci]
        ax.imshow(Image.open(r["path"]).convert("RGB"))
        match = r["pred"] == r["pred_class"]
        ax.set_title(f"label: {r['pred_class']}", color="green" if match else "red", fontsize=9)
plt.suptitle("Что модель относит к каждому классу (зелёный label = совпал с разметкой)", y=1.0)
plt.tight_layout(); plt.savefig(OUT_DIR / "val_predictions.png", dpi=110); plt.show()


In [ ]:
net.eval()
ckpt = {
    "state_dict": net.state_dict(),
    "arch": "mobilenetv3_small_100",
    "classes": CLASSES,
    "img_size": IMG_SIZE,
    "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
}
torch.save(ckpt, OUT_DIR / "photo_type_classifier.pth")

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
onnx_path = str(OUT_DIR / "photo_type_classifier.onnx")
onnx_kwargs = dict(
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
try:
    torch.onnx.export(net, dummy, onnx_path, dynamo=False, **onnx_kwargs)
except Exception as exc:
    print("legacy export не прошёл, ставлю onnxscript и пробую dynamo:", exc)
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxscript"], check=True)
    torch.onnx.export(net, dummy, onnx_path, dynamo=True, **onnx_kwargs)
json.dump(
    {"arch": "mobilenetv3_small_100", "classes": CLASSES, "img_size": IMG_SIZE,
     "normalize": ckpt["normalize"]},
    open(OUT_DIR / "photo_type_classifier.json", "w"), indent=2,
)
print("сохранено:", [p.name for p in OUT_DIR.glob("photo_type_classifier.*")])


In [ ]:
# Опциональная честная оценка: размень руками ~150-300 фото в gold.csv (filename,true_class).
# GOLD_CSV = "/kaggle/input/.../gold.csv"
# gold = pd.read_csv(GOLD_CSV)
# net.eval(); preds = []
# for fn in gold.filename:
#     img = Image.open(os.path.join(IMAGES_DIR, str(fn))).convert("RGB")
#     with torch.no_grad(), autocast():
#         out = net(val_tf(img).unsqueeze(0).to(device))
#     preds.append(CLASSES[out.argmax(1).item()])
# from sklearn.metrics import classification_report
# print(classification_report(gold.true_class, preds, labels=CLASSES, digits=3))
